## Code to query, visualize, and use output from the HRA Workflow Runner downstream

In [1]:
%pip install duckdb pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 78.1 MB/s eta 0:00:00:00:01

[notice] A new release of pip is available: 25.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
# Combine cell instances and normalize columns for analysis

import duckdb

data_dir = "data/gtex"
query = f"""
SELECT
  split_part(filename, '/', 3) as dataset,
  Organ_ID as organ,
  split_part(filename, '/', 4) as tool,
  column00 as cell,
  clid as cell_id,
  CL_Label as cell_label,
  match_type,
  COALESCE("mapping.score", conf_score, popv_prediction_score / 6, 0) as confidence_score,
FROM read_csv('
{ data_dir }/*/*/annotations.csv', 
union_by_name = true, 
filename = true, 
ignore_errors=true,
quote='"')
"""

cells = duckdb.sql(query)
cells.write_csv(f'{ data_dir }/cell-instances.csv.gz')
cells.show()

┌───────────────────────────────┬────────────────┬─────────┬─────────────────────────────────┬────────────┬─────────────────────────────────────┬─────────────────┬────────────────────┐
│            dataset            │     organ      │  tool   │              cell               │  cell_id   │             cell_label              │   match_type    │  confidence_score  │
│            varchar            │    varchar     │ varchar │             varchar             │  varchar   │               varchar               │     varchar     │       double       │
├───────────────────────────────┼────────────────┼─────────┼─────────────────────────────────┼────────────┼─────────────────────────────────────┼─────────────────┼────────────────────┤
│ GTEX-GTEX-12BJ1-5007-SM-H8L6U │ UBERON:0002367 │ popv    │ CST04_TATGCCCGTTCTGAAC-prostate │ CL:0002340 │ luminal cell of prostate epithelium │ skos:exactMatch │                1.0 │
│ GTEX-GTEX-12BJ1-5007-SM-H8L6U │ UBERON:0002367 │ popv    │ CST04_AGATTGCC

In [21]:
# Show only cells annotated with azimuth
cells.filter("tool = 'azimuth'")

┌───────────────────────────────┬──────────────────────┬─────────┬─────────────────────────────┬────────────────────┬────────────┬──────────────────────┬────────────────────┐
│            dataset            │        organ         │  tool   │            cell             │      cell_id       │ cell_label │      match_type      │  confidence_score  │
│            varchar            │       varchar        │ varchar │           varchar           │      varchar       │  varchar   │       varchar        │       double       │
├───────────────────────────────┼──────────────────────┼─────────┼─────────────────────────────┼────────────────────┼────────────┼──────────────────────┼────────────────────┤
│ GTEX-GTEX-13N11-5030-SM-H5JDW │ None                 │ azimuth │ CST03_TCTTTCCTCTGCTTGC-lung │ 0.935159738908882  │ None       │ AT2                  │ 1.0000000000000002 │
│ GTEX-GTEX-13N11-5030-SM-H5JDW │ None                 │ azimuth │ CST03_TCGTACCGTCAGCTAT-lung │ 0.9262700597571508 │ None   

In [22]:

# Cell summary of all cells (counted 3x because run with 3 annotation tools)
cells.aggregate("cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("cell_count DESC")

┌───────────────────────────────────────────┬──────────────────────────────────────────┬────────────┐
│                  cell_id                  │                cell_label                │ cell_count │
│                  varchar                  │                 varchar                  │   int64    │
├───────────────────────────────────────────┼──────────────────────────────────────────┼────────────┤
│ CL:0002063                                │ type II pneumocyte                       │      27364 │
│ CL:0000235                                │ macrophage                               │      22554 │
│ CL:0002131                                │ regular ventricular cardiac myocyte      │      21780 │
│ CL:0000057                                │ fibroblast                               │      18672 │
│ CL:0002062                                │ type I pneumocyte                        │      14423 │
│ CL:0000746                                │ cardiac muscle cell                 

In [37]:
# Cell summaries by tool
summaries = cells.aggregate("tool, cell_id, first(cell_label) as cell_label, count(cell) as cell_count").order("tool, cell_count DESC")
df = summaries.to_df()
df

,tool,cell_id,cell_label,cell_count
0,azimuth,CL:0002131,regular ventricular cardiac myocyte,15017
1,azimuth,CL:0002063,type II pneumocyte,9000
2,azimuth,CL:0000057,fibroblast,8342
3,azimuth,CL:0011031,monocyte-derived dendritic cell,6212
4,azimuth,CL:0002062,type I pneumocyte,5513
...,...,...,...,...
204,popv,CL:4028006,alveolar type 2 fibroblast cell,3
205,popv,CL:0000794,"CD8-positive, alpha-beta cytotoxic T cell",2
206,popv,CL:0002394,CD141-positive myeloid dendritic cell,1
207,popv,ASCTB-TEMP:cd8-positive-alpha-beta-t-cell,"CD8-positive, alpha-beta T cell",1


In [38]:
# Transform df to fit requirements for HRA US#2 at https://apps.humanatlas.io/us2
df['percentage'] = df['cell_count'].apply(lambda c: c/df['cell_count'].sum())

# add more required column
df['modality'] = 'sc_transcriptomics'

# rename columns as needed
df_renamed = df.rename(columns=
  {
    'cell_count':'count'
  }
)

# New column order
new_order = ['tool', 'modality', 'percentage', 'count', 'cell_label','cell_id']

# Reassign columns
df = df_renamed[new_order]

df

,tool,modality,percentage,count,cell_label,cell_id
0,azimuth,sc_transcriptomics,0.052530,15017,regular ventricular cardiac myocyte,CL:0002131
1,azimuth,sc_transcriptomics,0.031483,9000,type II pneumocyte,CL:0002063
2,azimuth,sc_transcriptomics,0.029181,8342,fibroblast,CL:0000057
3,azimuth,sc_transcriptomics,0.021730,6212,monocyte-derived dendritic cell,CL:0011031
4,azimuth,sc_transcriptomics,0.019285,5513,type I pneumocyte,CL:0002062
...,...,...,...,...,...,...
204,popv,sc_transcriptomics,0.000010,3,alveolar type 2 fibroblast cell,CL:4028006
205,popv,sc_transcriptomics,0.000007,2,"CD8-positive, alpha-beta cytotoxic T cell",CL:0000794
206,popv,sc_transcriptomics,0.000003,1,CD141-positive myeloid dendritic cell,CL:0002394
207,popv,sc_transcriptomics,0.000003,1,"CD8-positive, alpha-beta T cell",ASCTB-TEMP:cd8-positive-alpha-beta-t-cell


In [40]:
# Export to CSV
df.to_csv('output/cell_summary.csv')